In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

In [ ]:
file_path = str(PROJECT_ROOT / "Dataset Specs" / "hospital_occupancy_daily.csv")

df = pd.read_csv(file_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

daily_occupancy = df["occupied_beds"].asfreq("D")

print(daily_occupancy.head())
print(daily_occupancy.tail())

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(daily_occupancy.index, daily_occupancy.values)
plt.title("Daily Hospital Bed Occupancy")
plt.xlabel("Date")
plt.ylabel("Occupied Beds")
plt.tight_layout()
plt.show()

In [ ]:
rolling_mean = daily_occupancy.rolling(window=30).mean()
rolling_std = daily_occupancy.rolling(window=30).std()

plt.figure(figsize=(12, 5))
plt.plot(daily_occupancy, label="Daily Occupancy")
plt.plot(rolling_mean, label="30-day Rolling Mean")
plt.plot(rolling_std, label="30-day Rolling Standard Deviation")
plt.title("Hospital Occupancy: Rolling Statistics")
plt.ylabel("Occupied Beds")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(daily_occupancy)

print("ADF Statistic:", result[0])
print("p-value:", result[1])

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(12, 5))
plot_acf(daily_occupancy, ax=plt.gca(), lags=40)
plt.title("ACF: Daily Hospital Occupancy")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plot_pacf(daily_occupancy, ax=plt.gca(), lags=40, method="ywm")
plt.title("PACF: Daily Hospital Occupancy")
plt.tight_layout()
plt.show()

In [ ]:
train_size = int(len(daily_occupancy) * 0.7)

train = daily_occupancy.iloc[:train_size]
test = daily_occupancy.iloc[train_size:]

print("Total observations:", len(daily_occupancy))
print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training period:", train.index.min(), "to", train.index.max())
print("Testing period:", test.index.min(), "to", test.index.max())

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.figure(figsize=(12,5))
plot_acf(daily_occupancy, ax=plt.gca(), lags=40)
plt.title("ACF Plot")
plt.show()

plt.figure(figsize=(12,5))
plot_pacf(daily_occupancy, ax=plt.gca(), lags=40, method='ywm')
plt.title("PACF Plot")
plt.show()

In [ ]:
train_size = int(len(daily_occupancy) * 0.7)

train = daily_occupancy.iloc[:train_size]
test = daily_occupancy.iloc[train_size:]

print("Total observations:", len(daily_occupancy))
print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training period:", train.index.min(), "to", train.index.max())
print("Testing period:", test.index.min(), "to", test.index.max())

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train, order=(1, 0, 1))
model_fit = model.fit()

print(model_fit.summary())

In [ ]:
forecast = model_fit.get_forecast(steps=len(test))
y_pred = forecast.predicted_mean
y_pred.index = test.index

y_true = test

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("ARIMA(1,0,1) performance on the 70/30 test set")
print(f"RMSE: {rmse:.3f} beds")
print(f"MAE:  {mae:.3f} beds")
print(f"MAPE: {mape:.3f}%")

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(y_true.index, y_true, label="Actual Occupancy", color="black")
plt.plot(y_pred.index, y_pred, label="ARIMA Forecast", color="orange")

plt.title("ARIMA: Actual vs Forecast Hospital Occupancy (70/30 Split)")
plt.xlabel("Date")
plt.ylabel("Occupied Beds")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/70_30"
)
output_dir.mkdir(parents=True, exist_ok=True)

arima_df = pd.DataFrame({
    "date": y_true.index,
    "actual_occupied_beds": y_true.to_numpy(),
    "arima_predicted_occupied_beds": y_pred.to_numpy()
})

arima_df.to_csv(
    output_dir / "arima_occupancy_predictions.csv",
    index=False
)

print(f"ARIMA predictions saved to: {output_dir}")

In [ ]:
print("First 10 predictions:")
print(y_pred.head(10))

print("\nLast 10 predictions:")
print(y_pred.tail(10))

print("\nForecast standard deviation:", y_pred.std())
print("Unique rounded predictions:", y_pred.round(3).nunique())

In [ ]:
test_weekly = y_true.resample("W").mean()
pred_weekly = y_pred.resample("W").mean()

plt.figure(figsize=(12, 5))
plt.plot(test_weekly.index, test_weekly, label="Actual Weekly Mean")
plt.plot(pred_weekly.index, pred_weekly, label="Forecast Weekly Mean")

plt.title("ARIMA: Weekly Mean Hospital Occupancy (70/30 Test Period)")
plt.xlabel("Date")
plt.ylabel("Mean Occupied Beds")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()